# GardnAIze — Classification d'images de plantes
Bloc 5 : classification supervisée de photos de plantes, à partir d'un corpus d'espèces étiquetées.
**Statut :** squelette reproductible. Aucun dataset ni résultat d'entraînement n'est fourni ou inventé.
Configurer `DATASET_DIR` avec des images locales autorisées, réparties par classe. Ne pas committer les photos.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf

SEED = 42
tf.keras.utils.set_random_seed(SEED)
DATASET_DIR = Path(os.environ.get("DATASET_DIR", "../datasets/images/plants"))
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET_DIR.resolve())


## 1. Provenance et contrôle du dataset
Documenter la source, la licence, les classes, les doublons et les conditions d'utilisation dans `datasets/README.md`. Les images doivent être dans des sous-dossiers nommés par classe.
Classes envisagées (à confirmer selon les annotations réelles) : espèce_A, espèce_B.


In [ ]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Dataset absent : {DATASET_DIR}. Télécharger un dataset sous licence compatible.')
counts = {folder.name: len(list(folder.rglob('*.jpg'))) + len(list(folder.rglob('*.png'))) + len(list(folder.rglob('*.jpeg'))) for folder in DATASET_DIR.iterdir() if folder.is_dir()}
display(pd.DataFrame(sorted(counts.items()), columns=['classe', 'images']))
if len(counts) < 2 or min(counts.values()) < 3:
    raise ValueError('Il faut au moins deux classes et trois images par classe pour initialiser une expérience.')


## 2. Chargement et séparation
Séparation exploratoire 80/20 entraînement/validation. **Avant l'évaluation finale**, réserver un test indépendant, idéalement par source/jardin/plante, pour éviter la fuite de données et les quasi-doublons.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(DATASET_DIR, validation_split=0.2, subset='training', seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(DATASET_DIR, validation_split=0.2, subset='validation', seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
class_names = train_ds.class_names
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
print(class_names)


## 3. Baseline CNN et entraînement
Loss : categorical sparse cross-entropy ; optimisation Adam ; early stopping. Les scores de validation ne remplacent pas une évaluation finale sur un test indépendant.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])


In [ ]:
pd.DataFrame(history.history)[['loss', 'val_loss']].plot(title='Courbes de perte')
plt.show()
y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_pred = np.concatenate([np.argmax(model.predict(x, verbose=0), axis=1) for x, _ in val_ds])
print(classification_report(y_true, y_pred, labels=range(len(class_names)), target_names=class_names, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(len(class_names))), display_labels=class_names).plot(xticks_rotation=45)
plt.show()


## 4. Comparaison et déploiement (à compléter)
Comparer ce CNN avec MobileNetV2 (transfer learning), garder un **jeu de test indépendant**, consigner précision/rappel/F1 et erreurs, exporter un modèle `.keras` versionné hors Git, puis intégrer l'inférence à FastAPI avec tests et suivi de version. Aucun déploiement n'est réalisé par ce notebook.
